In [12]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

In [13]:
# Carregamento de Dados

nome_arquivo = 'datasets/dataset2.csv' 
df = pd.read_csv(nome_arquivo, sep=',') # Assumimndo separador vírgula

# Limpeza dos Nomes das Colunas
df.columns = df.columns.str.replace('#', '').str.strip().str.lower()

# Renomear colunas para o padrão do nosso modelo
df.rename(columns={'heart_rate': 'bpm', 
                   'activityid': 'target'}, 
          inplace=True)

# Filtragem aprimorada: Remove a atividade transitória (String)
df = df[df['target'] != 'transient activities'].copy()

# Codificação do Rótulo (Label Encoding)
le = LabelEncoder()
# Sobrescreve a coluna 'target' com os valores numéricos (0, 1, 2, 3...)
df['target'] = le.fit_transform(df['target']) 

# (lying, sitting, standing, tipicamente) e o restante é Atividade.
df['target'] = df['target'].apply(lambda x: 0 if x <= 2 else 1)

# Filtragem Essencial: Remove linhas com valores ausentes (NaN)
df.dropna(inplace=True)

print("--- Dados Carregados e Codificados ---")

--- Dados Carregados e Codificados ---


In [14]:
# Engenharia de Features

# Define o tamanho da janela (ex: 5 leituras passadas)
window_size = 5

# Média Móvel: Tendência central recente
df['rolling_mean'] = df['bpm'].rolling(window=window_size).mean()

# Desvio Padrão Móvel: Instabilidade/Variabilidade (CRUCIAL para sua demanda)
df['rolling_std'] = df['bpm'].rolling(window=window_size).std()

# Diferença do anterior (Aceleração do coração)
df['diff'] = df['bpm'].diff()

# Remove as primeiras linhas que ficaram NaN por causa do rolling
df.dropna(inplace=True)

In [15]:
# Preparação dos dados (validação será via StratifiedKFold abaixo)
features = ['bpm', 'rolling_mean', 'rolling_std', 'diff']
X = df[features]
y = df['target']
print(f'Prepared data: {X.shape[0]} samples, {X.shape[1]} features')
print('Class distribution:', y.value_counts().to_dict())

Prepared data: 1936477 samples, 4 features
Class distribution: {1: 1466554, 0: 469923}


In [16]:
# K-Fold evaluation with RandomForestClassifier
# Objetivo simulado: maximizar recall da classe Repouso (0) e minimizar falsos negativos dessa classe.
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, classification_report
from sklearn.calibration import CalibratedClassifierCV

# Prepare data (already prepared in previous cell)
features = ['bpm', 'rolling_mean', 'rolling_std', 'diff']
X = df[features]
y = df['target']

# K-Fold evaluation with RandomForestClassifier
# Objetivo simulado: maximizar recall da classe Repouso (0) e minimizar falsos negativos dessa classe.
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, classification_report
from sklearn.calibration import CalibratedClassifierCV

# Configuráveis
N_SPLITS = 5
N_ESTIMATORS = 200  # >= 200 para maior estabilidade
CLASS_WEIGHT = {0: 3, 1: 1}  # penaliza mais erros na classe Repouso (0)
THRESHOLD = 0.8  # P(Repouso) >= THRESHOLD => pred = 0
DO_CALIBRATION = False  # alterar para True para testar CalibratedClassifierCV
RANDOM_STATE = 42

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_metrics = []

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Cria e treina RandomForest mais conservador
    base_clf = RandomForestClassifier(n_estimators=N_ESTIMATORS, class_weight=CLASS_WEIGHT, random_state=RANDOM_STATE)
    if DO_CALIBRATION:
        clf = CalibratedClassifierCV(base_estimator=base_clf, cv=3)
    else:
        clf = base_clf

    clf.fit(X_train, y_train)

    # Probabilidades da classe Repouso (label 0)
    probs = clf.predict_proba(X_test) if hasattr(clf, 'predict_proba') else None
    if probs is None:
        # fallback para predict() se não houver probabilidades (não esperado para RF)
        preds = clf.predict(X_test)
    else:
        p_repouso = probs[:, 0]  # P(class=0)
        # Aplica threshold customizado: classifica como 0 (Repouso) apenas se P(Repouso) >= THRESHOLD
        preds = np.where(p_repouso >= THRESHOLD, 0, 1)

    # Métricas focadas em segurança (classe 0 = Repouso)
    recall_rep = recall_score(y_test, preds, pos_label=0)
    f1_rep = f1_score(y_test, preds, pos_label=0)
    cm = confusion_matrix(y_test, preds)
    acc = accuracy_score(y_test, preds)  # métrica secundária

    fold_metrics.append({'fold': fold_idx, 'recall_repouso': recall_rep, 'f1_repouso': f1_rep, 'accuracy': acc, 'confusion_matrix': cm.tolist()})

    print(f'Fold {fold_idx}: recall_repouso={recall_rep:.4f}, f1_repouso={f1_rep:.4f}, accuracy={acc:.4f}')
    print('Confusion matrix:\n', cm)
    print(classification_report(y_test, preds, target_names=['Repouso', 'Atividade/Anormal'], zero_division=0))

# Resumo agregado
import pandas as _pd
df_metrics = _pd.DataFrame(fold_metrics)
print('\n--- K-Fold Summary ---')
print(df_metrics[['fold','recall_repouso','f1_repouso','accuracy']])
print(f"Mean recall_repouso: {df_metrics['recall_repouso'].mean():.4f}, std: {df_metrics['recall_repouso'].std(ddof=1):.4f}")
print(f"Mean f1_repouso: {df_metrics['f1_repouso'].mean():.4f}, std: {df_metrics['f1_repouso'].std(ddof=1):.4f}")
print(f"Mean accuracy: {df_metrics['accuracy'].mean():.4f}, std: {df_metrics['accuracy'].std(ddof=1):.4f}")

# Treina modelo final em todos os dados para inferência exemplificativa (NÃO CLÍNICO)
final_base = RandomForestClassifier(n_estimators=N_ESTIMATORS, class_weight=CLASS_WEIGHT, random_state=RANDOM_STATE)
if DO_CALIBRATION:
    final_clf = CalibratedClassifierCV(base_estimator=final_base, cv=3)
else:
    final_clf = final_base

final_clf.fit(X, y)

# Inferência simulada usando mesmo threshold
sample = pd.DataFrame([[120, X['rolling_mean'].mean(), X['rolling_std'].mean(), 5]], columns=features)
probs = final_clf.predict_proba(sample) if hasattr(final_clf, 'predict_proba') else None
if probs is not None:
    p_rep = probs[0, 0]
    pred_label = 0 if p_rep >= THRESHOLD else 1
    print('\n--- Inferência Simulada ---')
    print('Dados de entrada:', sample.values)
    print(f'P(Repouso) = {p_rep:.3f} (threshold={THRESHOLD})')
    print('Predição final (0=Repouso,1=Atividade/Anormal):', pred_label)
    print('Nota: \"certeza\" é probabilidade estimada pelo modelo, não confiança clínica.')
else:
    print('Modelo não fornece probabilidades; não é possível aplicar threshold customizado.')

# Fim do pipeline - este notebook é apenas uma simulação e NÃO deve ser usado clinicamente.

Fold 1: recall_repouso=0.5110, f1_repouso=0.5998, accuracy=0.8345
Confusion matrix:
 [[ 48030  45955]
 [ 18149 275162]]
                   precision    recall  f1-score   support

          Repouso       0.73      0.51      0.60     93985
Atividade/Anormal       0.86      0.94      0.90    293311

         accuracy                           0.83    387296
        macro avg       0.79      0.72      0.75    387296
     weighted avg       0.83      0.83      0.82    387296

Fold 2: recall_repouso=0.5015, f1_repouso=0.5943, accuracy=0.8338
Confusion matrix:
 [[ 47136  46849]
 [ 17519 275792]]
                   precision    recall  f1-score   support

          Repouso       0.73      0.50      0.59     93985
Atividade/Anormal       0.85      0.94      0.90    293311

         accuracy                           0.83    387296
        macro avg       0.79      0.72      0.74    387296
     weighted avg       0.82      0.83      0.82    387296

Fold 3: recall_repouso=0.5101, f1_repouso=0.59